In [4]:
import pandas as pd


Evento, fecha de apertura y días a usar: lunes a viernes de tres semanas antes y tres después, sin festivos ni días atípicos.
Juntamos los archivos diarios de un evento en una sola tabla, con una fila por estación, día y hora.

In [5]:
evento = "calle72"
fecha_evento = pd.to_datetime("2026-08-22")
carpeta = "../data/raw/validaciones/calle72/"

fechas = [
    "20260803","20260804","20260805","20260806",
    "20260810","20260811","20260812",
    "20260813","20260814","20260818","20260819",
    "20260820","20260821","20260824","20260825","20260826",
    "20260827","20260828","20260831","20260901",
    "20260902","20260903","20260904","20260907","20260908",
    "20260909","20260910","20260911"
]

In [6]:
resumenes = []

for fecha in fechas:
    df = pd.read_csv(carpeta + "validacionTroncal" + fecha + ".zip",
                     usecols=["Fecha_Transaccion","Estacion_Parada","Linea"])

    df["Fecha_Transaccion"] = pd.to_datetime(df["Fecha_Transaccion"])
    df["fecha"] = df["Fecha_Transaccion"].dt.normalize()
    df["hora"] = df["Fecha_Transaccion"].dt.hour
    df = df[df["fecha"] == pd.to_datetime(fecha)]

    df["codigo"] = df["Estacion_Parada"].str.strip().str[1:6]

    resumen = (df.groupby(["fecha","codigo","hora"])
                 .agg(validaciones=("codigo","count"),
                      estacion=("Estacion_Parada","first"),
                      linea=("Linea","first"))
                 .reset_index())
    resumenes.append(resumen)

tabla = pd.concat(resumenes, ignore_index=True)
tabla.shape

(81191, 6)


Dejamos un solo nombre por código de estación y agregamos si el día es antes o después del evento.

In [7]:
nombres = tabla.groupby("codigo")["estacion"].first()
lineas = tabla.groupby("codigo")["linea"].first()
tabla["estacion"] = tabla["codigo"].map(nombres).str.strip().str[8:]
tabla["linea"] = tabla["codigo"].map(lineas).str.strip()

tabla["evento"] = evento
tabla["periodo"] = "antes"
tabla.loc[tabla["fecha"] >= fecha_evento,"periodo"] = "después"

dias = ["lunes","martes","miércoles","jueves","viernes","sábado","domingo"]
tabla["dia_semana"] = tabla["fecha"].dt.dayofweek.map(lambda d: dias[d])

tabla.head()

,fecha,codigo,hora,validaciones,estacion,linea,evento,periodo,dia_semana
0,2026-08-03,02000,4,584,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes
1,2026-08-03,02000,5,4584,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes
2,2026-08-03,02000,6,9262,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes
3,2026-08-03,02000,7,9362,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes
4,2026-08-03,02000,8,6139,Portal Norte - Unicervantes,(33) Zona B AutoNorte,calle72,antes,lunes


In [8]:
tabla["fecha"].nunique(), tabla.duplicated(["fecha","codigo","hora"]).sum()

(28, np.int64(0))

In [9]:
tabla.to_csv("../data/processed/validaciones_" + evento + ".csv", index=False)

## Tabla - Metro

In [ ]:
metro = pd.read_csv("../data/processed/estaciones_definitivas_caracas.csv")
metro = metro.drop(columns=["fuente"])
metro


,estacion_definitiva,vagones,estado_hoy,codigo_demanda,estacion_metro,integracion_oficial,nota
0,Hospital,NaN,no existe aún,NaN,E11 Hospital (HOMI),directa,Cerrada desde el 05/07/2025 sin temporal propi...
1,Tercer Milenio,NaN,no existe aún,NaN,NaN,ninguna,Cerrada desde el 12/04/2025 sin temporal propi...
2,Avenida Jiménez,NaN,temporal,9128.0,E12 Avenida Jiménez,directa,Temporal entre las calles 10 y 8 desde el 31/0...
3,Calle 19,NaN,no existe aún,NaN,NaN,ninguna,Cerrada desde el 08/03/2025 sin temporal propi...
4,Calle 22,NaN,temporal,9129.0,NaN,ninguna,Temporal entre las calles 19 y 22 desde el 22/...
5,Calle 26 - Atrio,2.0,definitiva abierta,9114.0,E13 Central,proximidad,Completa desde el 20/06/2026. Integración en d...
6,Calle 39,NaN,temporal,9130.0,NaN,ninguna,Temporal Av. 39 entre la diagonal 40 y la call...
7,Calle 45 - American School Way,NaN,no existe aún,NaN,E14 Calle 45,directa,Cerrada desde el 18/04/2026 sin temporal propi...
8,Calle 53 - Universidad Manuela Beltrán,3.0,definitiva abierta,9131.0,NaN,ninguna,Abierta desde el 24/10/2026; reemplaza a Marly...
9,Calle 57,NaN,temporal,9125.0,NaN,ninguna,La EMB (2025) dijo que la temporal operaría mi...


Notas

Calle 53 aparece en los datos como Marly - Calle 57 (código 09131). No usamos Hora_Pico_SN porque marca Peak Time en todas las filas.

## Validaciones agrupadas por hora - calle 72, calle 53 y calle 26
- estacion = nombre de la estacion.
- carpeta_cruda = localizacion del archivo a procesar.
- carpeta_salida = localizacion del archivo ya procesado.
- acceso_norte y acceso_sur = localizacion del torniquete en el que se hizo la validacion.
- fechas = fechas DESPUES de la apertura de la estacion (osea cuando ya este operando).

Diccionario con las opciones para la transformación de archivos:

In [6]:
opciones = {
    "calle72": {
        "estacion": "(09122) Calle 72 - Areandina",
        "carpeta_cruda": "../data/raw/validaciones/calle72/",
        "carpeta_salida": "../data/processed/calle72-diarias/",
        "fechas": ["20260824", "20260825", "20260826", "20260827", "20260828",
                   "20260831", "20260901", "20260902", "20260903", "20260904",
                   "20260907", "20260908", "20260909", "20260910", "20260911"],
    },
    "calle26": {
        "estacion": "(09114) Calle 26",
        "carpeta_cruda": "../data/raw/validaciones/calle26/",
        "carpeta_salida": "../data/processed/calle26-diarias/",
        "acceso_norte": "(AA) Acceso Norte",
        "acceso_sur": "(BA) Acceso Sur",
        "fechas": ["20260622", "20260623", "20260624", "20260625", "20260626",
                   "20260630", "20260701", "20260702", "20260703",
                   "20260706", "20260707", "20260708", "20260709", "20260710"],
    },
    "calle53": {
        "estacion": "(09131) Marly - Calle 57",
        "carpeta_cruda": "../data/raw/validaciones/calle53/",
        "carpeta_salida": "../data/processed/calle53-diarias/",
        "acceso_norte": "(05) Acceso Norte",
        "acceso_sur": "(03) Acceso Sur",
        "fechas": ["20261026", "20261027", "20261028", "20261029", "20261030",
                   "20261103", "20261104", "20261105", "20261106",
                   "20261109", "20261110", "20261111", "20261112", "20261113"],
    },
}

Transformación de archivos:

Opciones: calle72, calle53 y calle 26.

In [8]:
opcion = opciones["calle26"]
fechas = opcion["fechas"]

for fecha in fechas:
    df = pd.read_csv(opcion["carpeta_cruda"] + "validacionTroncal" + fecha + ".zip")
    df = df[(df["Estacion_Parada"] == opcion["estacion"])]
    df["Fecha_Transaccion"] = pd.to_datetime(df["Fecha_Transaccion"])
    df["hora"] = df["Fecha_Transaccion"].dt.hour
    df_norte = df[(df["Acceso_Estacion"] == opcion["acceso_norte"])]
    df_sur = df[(df["Acceso_Estacion"] == opcion["acceso_sur"])]

    df_norte = (
        df_norte.groupby("hora")
        .agg(validaciones=("Estacion_Parada", "count"),
             dispositivos=("Dispositivo", "nunique"))
    )

    df_sur = (
        df_sur.groupby("hora")
        .agg(validaciones=("Estacion_Parada", "count"),
             dispositivos=("Dispositivo", "nunique"))
    )
    df_norte.to_csv(opcion["carpeta_salida"] + "validaciones_norte_" + fecha + ".csv")
    df_sur.to_csv(opcion["carpeta_salida"] + "validaciones_sur_" + fecha + ".csv")

## Retornar 2 tablas: Acceso Sur y Acceso Norte por estación.

Se promedian todos las validaciones agrupadas por hora y por estacion de un acceso en especifico (Norte o Sur).

In [11]:
opcion = opciones["calle26"]
fechas = opcion["fechas"]

tablas_norte = []
tablas_sur = []

for fecha in fechas:
    norte = pd.read_csv(opcion["carpeta_salida"] + "validaciones_norte_" + fecha + ".csv")
    norte["fecha"] = fecha
    tablas_norte.append(norte)

    sur = pd.read_csv(opcion["carpeta_salida"] + "validaciones_sur_" + fecha + ".csv")
    sur["fecha"] = fecha
    tablas_sur.append(sur)

norte_todos_los_dias = pd.concat(tablas_norte, ignore_index=True)
sur_todos_los_dias = pd.concat(tablas_sur, ignore_index=True)

promedio_norte = (norte_todos_los_dias.groupby("hora")
                  .agg(validaciones_promedio=("validaciones", "mean"),
                       validaciones_maximo=("validaciones", "max"),
                       dispositivos_promedio=("dispositivos", "mean"),
                       dias=("fecha", "nunique"))
                  .reset_index())

promedio_norte["validaciones_promedio"] = promedio_norte["validaciones_promedio"].round(0)
promedio_norte["dispositivos_promedio"] = promedio_norte["dispositivos_promedio"].round(1)
promedio_norte.to_csv(opcion["carpeta_salida"] + "promedio_por_hora_norte.csv", index=False)

promedio_sur = (sur_todos_los_dias.groupby("hora")
                .agg(validaciones_promedio=("validaciones", "mean"),
                     validaciones_maximo=("validaciones", "max"),
                     dispositivos_promedio=("dispositivos", "mean"),
                     dias=("fecha", "nunique"))
                .reset_index())

promedio_sur["validaciones_promedio"] = promedio_sur["validaciones_promedio"].round(0)
promedio_sur["dispositivos_promedio"] = promedio_sur["dispositivos_promedio"].round(1)
promedio_sur.to_csv(opcion["carpeta_salida"] + "promedio_por_hora_sur.csv", index=False)